# Inferencia en tiempo real con webcam

Ejecutar en Colab después de entrenar el modelo en el notebook anterior.

In [ ]:
# Instalar dependencias
!pip install -q ultralytics opencv-python-headless

In [ ]:
# Cargar modelo entrenado
from ultralytics import YOLO
import cv2
import numpy as np
from IPython.display import display, Image
from google.colab.patches import cv2_imshow
from google.colab import output
import base64
from io import BytesIO
from PIL import Image as PILImage

# Cargar el mejor modelo del entrenamiento
model = YOLO('runs/train/kortxovision_yolov8s/weights/best.pt')
print("Modelo cargado correctamente")

In [ ]:
# Funciones para captura de webcam
from IPython.display import Javascript, display
from google.colab.output import eval_js

def take_photo(filename='photo.jpg', quality=0.8):
    js = Javascript('''
        async function takePhoto(quality) {
            const div = document.createElement('div');
            const capture = document.createElement('button');
            capture.textContent = 'Capturar';
            div.appendChild(capture);

            const video = document.createElement('video');
            video.style.display = 'block';
            const stream = await navigator.mediaDevices.getUserMedia({video: true});

            document.body.appendChild(div);
            div.appendChild(video);
            video.srcObject = stream;
            await video.play();

            // Resize video
            google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

            // Wait for click
            await new Promise((resolve) => capture.onclick = resolve);

            const canvas = document.createElement('canvas');
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            canvas.getContext('2d').drawImage(video, 0, 0);
            stream.getVideoTracks()[0].stop();
            div.remove();
            return canvas.toDataURL('image/jpeg', quality);
        }
    ''')
    display(js)
    data = eval_js('takePhoto({})'.format(quality))
    return data

print("Funciones de webcam listas")

In [ ]:
# Captura y detección en tiempo real
import time
from IPython.display import clear_output

try:
    while True:
        # Capturar frame
        print("Capturando desde webcam...")
        image_data = take_photo(quality=0.8)
        
        # Convertir de base64 a imagen
        image_bytes = base64.b64decode(image_data.split(',')[1])
        img = PILImage.open(BytesIO(image_bytes))
        
        # Realizar detección
        results = model(img)
        
        # Mostrar resultado con bounding boxes
        clear_output(wait=True)
        annotated = results[0].plot()
        annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
        
        # Mostrar imagen
        display(PILImage.fromarray(annotated_rgb))
        
        # Mostrar detecciones
        if len(results[0].boxes) > 0:
            print(f"\nDetecciones encontradas: {len(results[0].boxes)}")
            for box in results[0].boxes:
                cls = int(box.cls[0])
                conf = float(box.conf[0])
                print(f"- {model.names[cls]}: {conf:.2f}")
        else:
            print("\nNo se detectaron objetos")
        
        time.sleep(0.1)
        
except KeyboardInterrupt:
    print("\nDetención manual")